In [1]:
from sqlalchemy import create_engine
import pandas as pd
import plotly.express as px

PROJECT_ID = "sg-20min-town-mod2proj"
engine = create_engine(f"bigquery://{PROJECT_ID}/olist_marts")
print("Connected!")

Connected!


In [2]:
sql = """
    SELECT
        p.product_category_english              AS category,
        COUNT(DISTINCT f.order_id)              AS order_count,
        ROUND(SUM(f.total_sale_amount), 2)      AS total_revenue,
        ROUND(AVG(f.total_sale_amount), 2)      AS avg_order_value,
        ROUND(AVG(f.review_score), 2)           AS avg_review_score,
        ROUND(AVG(f.delivery_days), 1)          AS avg_delivery_days,
        COUNTIF(f.is_late_delivery) * 100.0
            / COUNT(*)                          AS late_pct
    FROM olist_marts.fact_orders      f
    JOIN olist_marts.dim_product      p  ON f.product_id = p.product_id
    WHERE f.order_status = 'delivered'
      AND p.product_category_english IS NOT NULL
    GROUP BY category
    HAVING order_count >= 100
    ORDER BY total_revenue DESC
    LIMIT 15
"""
df = pd.read_sql(sql, engine)
df

,category,order_count,total_revenue,avg_order_value,avg_review_score,avg_delivery_days,late_pct
0,health_beauty,8647,1412089.53,149.19,4.19,11.9,7.564712
1,watches_gifts,5495,1264333.12,215.79,4.07,12.6,7.202594
2,bed_bath_table,9272,1225209.26,111.86,3.92,12.8,7.030037
3,sports_leisure,7530,1118256.91,132.64,4.17,12.1,6.310046
4,computers_accessories,6530,1032723.77,135.10,3.99,13.2,6.488749
5,furniture_decor,6307,880329.92,107.88,3.95,12.8,7.034314
6,housewares,5743,758392.25,111.61,4.11,10.9,5.003679
7,cool_stuff,3559,691680.89,186.04,4.20,12.3,5.836471
8,auto,3810,669454.75,161.70,4.12,12.2,7.028986
9,garden_tools,3448,567145.68,132.88,4.08,13.7,6.560450


In [3]:
top10 = df.head(10).sort_values("total_revenue")
fig = px.bar(top10, x="total_revenue", y="category", orientation="h",
    title="Top 10 Categories by Revenue",
    color="avg_review_score", color_continuous_scale="RdYlGn")
fig.show()
fig.write_image("../docs/chart_02_top_categories.png", width=1200, height=600)
print("Saved chart_02_top_categories.png")

Saved chart_02_top_categories.png
